In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

In [2]:
INFERENCE_DIR = Path("/storage/project/ps-clef2026_plantclef-0/shared/inference/dinov2-vit-base")
NUM_CLASSES = 7806

files = sorted(INFERENCE_DIR.glob("*.parquet"))
print(f"{len(files)} parquet files:")
for f in files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:25s} {size_mb:7.1f} MB")

10 parquet files:
  grid_10x10.parquet         3133.8 MB
  grid_2x2.parquet            125.4 MB
  grid_3x3.parquet            282.1 MB
  grid_4x4.parquet            501.4 MB
  grid_5x5.parquet            783.5 MB
  grid_6x6.parquet           1128.1 MB
  grid_7x7.parquet           1535.6 MB
  grid_8x8.parquet           2005.5 MB
  grid_9x9.parquet           2538.3 MB
  no_grid.parquet              31.4 MB


In [3]:
# Load 2x2 grid parquet
df = pq.read_table(INFERENCE_DIR / "grid_2x2.parquet").to_pandas()
df["probs_array"] = df["probs"].apply(lambda b: np.frombuffer(b, dtype=np.float16))
print(f"Shape: {df.shape}")
print(f"Unique images: {df['image_name'].nunique()}")
print(f"Tiles per image: {df.groupby('image_name').size().unique()}")
df[["image_name", "tile_idx", "probs_array"]]

Shape: (8420, 4)
Unique images: 2105
Tiles per image: [4]


,image_name,tile_idx,probs_array
0,2024-CEV3-20240602,0,"[5.5e-06, 1.377e-05, 3.04e-06, 2.9e-06, 1.55e-..."
1,2024-CEV3-20240602,1,"[2.9e-05, 2.587e-05, 5e-06, 2.396e-05, 6.133e-..."
2,2024-CEV3-20240602,2,"[6.74e-06, 4.816e-05, 1.067e-05, 2.11e-05, 1.9..."
3,2024-CEV3-20240602,3,"[1.12e-05, 7.7e-06, 1.585e-05, 7.9e-06, 2.54e-..."
4,CBN-PdlC-A1-20130807,0,"[1.03e-05, 7.8e-06, 9.3e-06, 1.29e-05, 6.4e-06..."
...,...,...,...
8415,RNNB-8-8-20240118,3,"[3.18e-05, 7.37e-05, 3.237e-05, 1.734e-05, 4.5..."
8416,RNNB-8-9-20240118,0,"[3.21e-05, 8.607e-05, 0.0001479, 0.00010127, 3..."
8417,RNNB-8-9-20240118,1,"[4.95e-06, 0.0001416, 2.44e-05, 4.35e-05, 2.38..."
8418,RNNB-8-9-20240118,2,"[7e-06, 2.9e-05, 7.17e-05, 3.32e-05, 1.085e-05..."


In [4]:
# Sanity check: probs vector length and softmax sum
sample = df.iloc[0]["probs_array"]
print(f"Probs length: {len(sample)} (expected {NUM_CLASSES})")
print(f"Sum: {sample.sum():.4f}")
print(f"Min: {sample.min():.6f}, Max: {sample.max():.6f}")
print(f"Top-5 class indices: {np.argsort(sample)[-5:][::-1]}")

Probs length: 7806 (expected 7806)
Sum: 1.0000
Min: 0.000000, Max: 0.141479
Top-5 class indices: [5516 4418 4403 1444 6262]


In [5]:
# Verify all tiles sum to ~1.0
sums = df["probs_array"].apply(lambda p: p.sum())
print(f"Row sums — min: {sums.min():.4f}, max: {sums.max():.4f}, mean: {sums.mean():.4f}")

Row sums — min: 0.9995, max: 1.0000, mean: 1.0000
